In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Mounted at /content/drive
root: /content/drive/MyDrive/CALSHIFT_Research/calshift-research
seeds: [42, 1337, 2024, 7, 91, 512, 6021, 88, 3407, 12345]


In [2]:
# =============================================================================
# Cell 2 - PARTITION SEED
# Distinct from the ten model seeds. The source partition is drawn ONCE and
# held fixed across all model seeds.
#
# Rationale, recorded: calibration-set variability is already supplied by the
# R = 10 matched draws (preregistration 8.1). Re-drawing the source partition
# per model seed would confound partition draw with model initialisation and
# would double-count calibration variability. Model seeds therefore vary model
# training only.
# =============================================================================
PARTITION_SEED = 20260724

DECISION = {
    'partition_seed': PARTITION_SEED,
    'source_partition_redrawn_per_model_seed': False,
    'rationale': ('matched draws supply calibration variability; re-drawing the '
                  'source partition per seed would confound partition draw with '
                  'model initialisation'),
    'stratification': 'broad class (label); subtype composition recorded not enforced',
}
print(json.dumps(DECISION, indent=2))
print('\nADD THIS TO preregistration.md IF NOT ALREADY PRESENT')

{
  "partition_seed": 20260724,
  "source_partition_redrawn_per_model_seed": false,
  "rationale": "matched draws supply calibration variability; re-drawing the source partition per seed would confound partition draw with model initialisation",
  "stratification": "broad class (label); subtype composition recorded not enforced"
}

ADD THIS TO preregistration.md IF NOT ALREADY PRESENT


In [3]:
# =============================================================================
# Cell 3 - load interim data from notebook 01
# =============================================================================
nsl_train = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet')
nsl_test  = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet')

nsl_train = nsl_train.reset_index(drop=True)
nsl_test  = nsl_test.reset_index(drop=True)

print('source pool (KDDTrain+):', nsl_train.shape)
print('target pool (KDDTest+) :', nsl_test.shape)
assert len(nsl_train) == 125973, 'unexpected source row count'
assert len(nsl_test)  == 22544,  'unexpected target row count'

source pool (KDDTrain+): (125973, 43)
target pool (KDDTest+) : (22544, 43)


In [4]:
# =============================================================================
# Cell 4 - stratified source split into the four source partitions
# Exact proportions per class. Rounding remainder goes to the largest partition
# so that every row is assigned exactly once.
# =============================================================================
def stratified_split(df, fractions, seed, stratify_col='label'):
    rng = np.random.default_rng(seed)
    names = list(fractions.keys())
    fracs = np.array([fractions[k] for k in names], dtype=float)
    assert abs(fracs.sum() - 1.0) < 1e-9, 'fractions must sum to 1'
    largest = names[int(np.argmax(fracs))]
    assign = pd.Series(index=df.index, dtype=object)

    for cls, sub in df.groupby(stratify_col, sort=True):
        idx = sub.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        counts = np.floor(fracs * n).astype(int)
        counts[names.index(largest)] += n - counts.sum()
        start = 0
        for name, c in zip(names, counts):
            assign.loc[idx[start:start + c]] = name
            start += c
        assert start == n
    return assign

split = stratified_split(nsl_train, config.SPLIT_FRACTIONS, PARTITION_SEED)
nsl_train = nsl_train.assign(partition=split.values)

D_train   = nsl_train[nsl_train.partition == 'train']
D_val     = nsl_train[nsl_train.partition == 'val']
D_probcal = nsl_train[nsl_train.partition == 'probcal']
S_pool    = nsl_train[nsl_train.partition == 'source_cal_pool']

for name, part in [('D_train', D_train), ('D_val', D_val),
                   ('D_probcal', D_probcal), ('S_pool', S_pool)]:
    print(f'{name:10s} {len(part):7d}  {len(part)/len(nsl_train):6.3f}')
print(f'{"total":10s} {len(nsl_train):7d}')

D_train      75590   0.600
D_val        12595   0.100
D_probcal    18894   0.150
S_pool       18894   0.150
total       125973


In [5]:
# =============================================================================
# Cell 5 - partition integrity assertions
# These must pass. A failure here invalidates every downstream result.
# =============================================================================
parts = {'train': D_train, 'val': D_val, 'probcal': D_probcal, 'source_cal_pool': S_pool}

total = sum(len(p) for p in parts.values())
assert total == len(nsl_train), f'row loss: {total} != {len(nsl_train)}'

idx_sets = {k: set(v.index) for k, v in parts.items()}
for a in idx_sets:
    for b in idx_sets:
        if a < b:
            overlap = idx_sets[a] & idx_sets[b]
            assert not overlap, f'OVERLAP between {a} and {b}: {len(overlap)} rows'

union = set().union(*idx_sets.values())
assert union == set(nsl_train.index), 'partition union does not cover the source pool'

print('no row loss')
print('all partitions pairwise disjoint')
print('union covers source pool')

prev = pd.DataFrame({k: v['label'].value_counts(normalize=True) for k, v in parts.items()})
prev['source_overall'] = nsl_train['label'].value_counts(normalize=True)
print()
print((prev * 100).round(3).to_string())
max_dev = (prev.drop(columns='source_overall')
             .sub(prev['source_overall'], axis=0).abs().max().max())
print(f'\nmax stratification deviation: {max_dev*100:.4f} percentage points')
assert max_dev < 0.01, 'stratification deviates more than 1 percentage point'

no row loss
all partitions pairwise disjoint
union covers source pool

         train     val  probcal  source_cal_pool  source_overall
label                                                           
Normal  53.455  53.466   53.461           53.461          53.458
DoS     36.456  36.459   36.461           36.461          36.458
Probe    9.254   9.250    9.252            9.252           9.253
R2L      0.791   0.786    0.789            0.789           0.790
U2R      0.044   0.040    0.037            0.037           0.041

max stratification deviation: 0.0074 percentage points


In [6]:
# =============================================================================
# Cell 6 - realised calibration counts
# SHC calibrates on S_pool. TSC calibrates on T_cal drawn from the target pool
# at the ladder D_eval size fixed in Amendment 1 (2,340 rows, natural
# KDDTest+ prevalence, |T_cal| = |D_eval|).
# =============================================================================
D_EVAL_SIZE = 2340   # Amendment 1, section A3.2

tgt_prev = nsl_test['label'].value_counts(normalize=True)
tcal_projected = (tgt_prev * D_EVAL_SIZE).round().astype(int)

shc_counts = S_pool['label'].value_counts()

calib = pd.DataFrame({
    'shc_realised': shc_counts,
    'tsc_at_rung':  tcal_projected,
}).reindex(config.CANONICAL_CLASSES).fillna(0).astype(int)
calib['target_pool_total'] = nsl_test['label'].value_counts().reindex(config.CANONICAL_CLASSES).fillna(0).astype(int)

print('calibration counts available per class')
print(calib.to_string())
print(f'\nS_pool total: {len(S_pool)}   T_cal per rung: {D_EVAL_SIZE}')

calibration counts available per class
        shc_realised  tsc_at_rung  target_pool_total
label                                               
Normal         10101         1008               9711
DoS             6889          774               7460
Probe           1748          251               2421
R2L              149          299               2885
U2R                7            7                 67

S_pool total: 18894   T_cal per rung: 2340


In [7]:
# =============================================================================
# Cell 7 - BINDING feasibility table
# Supersedes the projection in notebook 01 cell 13.
# A class supports the paired TSC-vs-SHC contrast only if BOTH calibration
# sources satisfy n_c >= ceil(1/alpha) - 1.
# =============================================================================
alphas = [config.ALPHA_PRIMARY] + config.ALPHA_SENSITIVITY + config.ALPHA_CONDITIONAL

rows = []
for a in alphas:
    need = config.min_calib_n(a)
    for cls in config.CANONICAL_CLASSES:
        s = int(calib.loc[cls, 'shc_realised'])
        t = int(calib.loc[cls, 'tsc_at_rung'])
        rows.append({
            'dataset': 'nslkdd', 'alpha': a, 'class': cls,
            'min_calib_needed': need,
            'shc_realised': s, 'tsc_at_rung': t,
            'shc_feasible': s >= need, 'tsc_feasible': t >= need,
            'contrast_possible': (s >= need) and (t >= need),
            'asymmetric': (t >= need) and (s < need),
        })

feas = pd.DataFrame(rows)
feas.to_csv(config.REPORTS_DIR / 'feasibility_binding_nslkdd.csv', index=False)
print(feas.to_string(index=False))

asym = feas[feas.asymmetric]
if len(asym):
    print('\nASYMMETRIC (TSC feasible, SHC not) - paired contrast impossible:')
    print(asym[['alpha', 'class', 'shc_realised', 'tsc_at_rung',
                'min_calib_needed']].to_string(index=False))

dataset  alpha  class  min_calib_needed  shc_realised  tsc_at_rung  shc_feasible  tsc_feasible  contrast_possible  asymmetric
 nslkdd   0.05 Normal                19         10101         1008          True          True               True       False
 nslkdd   0.05    DoS                19          6889          774          True          True               True       False
 nslkdd   0.05  Probe                19          1748          251          True          True               True       False
 nslkdd   0.05    R2L                19           149          299          True          True               True       False
 nslkdd   0.05    U2R                19             7            7         False         False              False       False
 nslkdd   0.10 Normal                 9         10101         1008          True          True               True       False
 nslkdd   0.10    DoS                 9          6889          774          True          True               True     

In [8]:
# =============================================================================
# Cell 8 - focal class, BINDING
# Rule (preregistration 12): the rarest attack class supporting the paired
# contrast in the SOURCE calibration pool at the primary alpha.
# Written once. Never revised after any coverage number exists.
# =============================================================================
attacks = [c for c in config.CANONICAL_CLASSES if c != 'Normal']
elig = feas[(feas.alpha == config.ALPHA_PRIMARY) &
            (feas.contrast_possible) &
            (feas['class'].isin(attacks))].copy()

assert len(elig) > 0, 'NO FEASIBLE FOCAL CLASS at primary alpha - design must be revisited'

elig = elig.sort_values('shc_realised')
FOCAL_CLASS = str(elig.iloc[0]['class'])

excluded = feas[(feas.alpha == config.ALPHA_PRIMARY) &
                (~feas.contrast_possible) &
                (feas['class'].isin(attacks))]['class'].tolist()

record = {
    'dataset': 'nslkdd',
    'focal_class': FOCAL_CLASS,
    'alpha_primary': config.ALPHA_PRIMARY,
    'rule': ('rarest attack class supporting the paired TSC-vs-SHC contrast '
             'in the source calibration pool at the primary alpha'),
    'shc_realised_counts': {c: int(calib.loc[c, 'shc_realised'])
                            for c in config.CANONICAL_CLASSES},
    'tsc_counts_at_rung': {c: int(calib.loc[c, 'tsc_at_rung'])
                           for c in config.CANONICAL_CLASSES},
    'min_calib_needed': config.min_calib_n(config.ALPHA_PRIMARY),
    'excluded_attack_classes': excluded,
    'partition_seed': PARTITION_SEED,
    'd_eval_size': D_EVAL_SIZE,
    'status': 'BINDING - no coverage number has been computed at time of writing',
}

focal_path = config.REPORTS_DIR / 'focal_class_record.json'
if focal_path.exists():
    prior = json.loads(focal_path.read_text())
    if prior.get('focal_class') != FOCAL_CLASS:
        raise RuntimeError(
            f"focal class already recorded as {prior.get('focal_class')}, "
            f"now computes as {FOCAL_CLASS}. Do not overwrite. Log a deviation.")
    print('focal class already recorded and unchanged')
else:
    focal_path.write_text(json.dumps(record, indent=2))
    print('focal class RECORDED')

print(json.dumps(record, indent=2))

focal class RECORDED
{
  "dataset": "nslkdd",
  "focal_class": "R2L",
  "alpha_primary": 0.05,
  "rule": "rarest attack class supporting the paired TSC-vs-SHC contrast in the source calibration pool at the primary alpha",
  "shc_realised_counts": {
    "Normal": 10101,
    "DoS": 6889,
    "Probe": 1748,
    "R2L": 149,
    "U2R": 7
  },
  "tsc_counts_at_rung": {
    "Normal": 1008,
    "DoS": 774,
    "Probe": 251,
    "R2L": 299,
    "U2R": 7
  },
  "min_calib_needed": 19,
  "excluded_attack_classes": [
    "U2R"
  ],
  "partition_seed": 20260724,
  "d_eval_size": 2340,
  "status": "BINDING - no coverage number has been computed at time of writing"
}


In [9]:
# =============================================================================
# Cell 9 - deviation check against the notebook 01 projection
# NOTE: only SHC is comparable. Notebook 01 projected T_cal as half the whole
# target pool; Amendment 1 subsequently fixed D_eval (and therefore T_cal) at
# 2,340 rows. The TSC columns are different quantities by construction and a
# difference there is NOT a deviation.
# =============================================================================
proj_path = config.REPORTS_DIR / 'feasibility_projected_nslkdd.csv'
deviations = []

if proj_path.exists():
    proj = pd.read_csv(proj_path)
    m = proj.merge(feas, on=['dataset', 'alpha', 'class'], suffixes=('_proj', '_real'))

    shc_diff = m[m.shc_feasible_proj != m.shc_feasible_real]
    if len(shc_diff):
        deviations.append({
            'item': 'SHC feasibility changed between projection and realisation',
            'detail': shc_diff[['alpha', 'class', 'shc_calib_projected',
                                'shc_realised']].to_dict('records'),
        })
        print('DEVIATION: SHC feasibility changed')
        print(shc_diff[['alpha', 'class', 'shc_calib_projected', 'shc_realised',
                        'shc_feasible_proj', 'shc_feasible_real']].to_string(index=False))
    else:
        print('SHC feasibility matches the notebook 01 projection')

    gap = (m['shc_realised'] - m['shc_calib_projected']).abs().max()
    print(f'max |realised - projected| SHC count: {gap}')
    print('TSC columns intentionally not compared (Amendment 1 redefined T_cal size)')
else:
    print('no projection found; skipping comparison')

if deviations:
    dev_path = config.REPORTS_DIR / 'deviations.md'
    with open(dev_path, 'a') as f:
        f.write(f'\n## notebook 02\n```json\n{json.dumps(deviations, indent=2)}\n```\n')
    print('logged to reports/deviations.md')

SHC feasibility matches the notebook 01 projection
max |realised - projected| SHC count: 1
TSC columns intentionally not compared (Amendment 1 redefined T_cal size)


In [10]:
# =============================================================================
# Cell 10 - persist partitions and their fingerprints
# Index arrays go to data/ (gitignored). Fingerprints go to reports/ (committed)
# so a third party can verify the partition without the data.
# =============================================================================
config.PROC_DIR.mkdir(parents=True, exist_ok=True)

def fingerprint(idx):
    arr = np.sort(np.asarray(idx, dtype=np.int64))
    return hashlib.sha256(arr.tobytes()).hexdigest()

fps = {}
for name, part in parts.items():
    np.save(config.PROC_DIR / f'nslkdd_source_{name}_idx.npy',
            np.sort(part.index.to_numpy()))
    fps[f'source/{name}'] = {'n': int(len(part)), 'sha256': fingerprint(part.index)}

nsl_train[['partition']].to_parquet(
    config.PROC_DIR / 'nslkdd_source_partition_labels.parquet')

fp_payload = {
    'partition_seed': PARTITION_SEED,
    'split_fractions': config.SPLIT_FRACTIONS,
    'stratify_col': 'label',
    'fingerprints': fps,
}
(config.REPORTS_DIR / 'partition_fingerprints.json').write_text(
    json.dumps(fp_payload, indent=2))

print(json.dumps(fp_payload, indent=2))

{
  "partition_seed": 20260724,
  "split_fractions": {
    "train": 0.6,
    "val": 0.1,
    "probcal": 0.15,
    "source_cal_pool": 0.15
  },
  "stratify_col": "label",
  "fingerprints": {
    "source/train": {
      "n": 75590,
      "sha256": "2bbed545bfb1773a11f0f4fc203376022ac24cb17c1d50354f57355a2ea49a7d"
    },
    "source/val": {
      "n": 12595,
      "sha256": "2240be60741216772ac3366462a0d2b3705d4228bcccd0bc3409f00e1bc8ad41"
    },
    "source/probcal": {
      "n": 18894,
      "sha256": "ec2dcdecbb67637121131bf897cd4ffc2e56df96d58229c75a3e293ef3c62d91"
    },
    "source/source_cal_pool": {
      "n": 18894,
      "sha256": "6bdea16ee00fa27e15be6d04e7d82de022190fc2424116c12070e747362a4e99"
    }
  }
}


In [11]:
# =============================================================================
# Cell 11 - subtype composition of the source calibration pool
# Recorded, not enforced. Stratification is by broad class, so rare-class
# subtype mix in S_pool is random. This table makes that visible.
# =============================================================================
sub_tbl = (S_pool.groupby(['label', 'subtype']).size()
           .rename('n_in_S_pool').reset_index()
           .sort_values(['label', 'n_in_S_pool'], ascending=[True, False]))
sub_tbl.to_csv(config.REPORTS_DIR / 'nslkdd_Spool_subtype_composition.csv', index=False)

print(sub_tbl.to_string(index=False))

focal_sub = (S_pool[S_pool['label'] == FOCAL_CLASS]
             .groupby('subtype').size().sort_values(ascending=False))
if len(focal_sub):
    top = focal_sub.iloc[0]; tot = focal_sub.sum()
    print(f'\nFOCAL CLASS ({FOCAL_CLASS}) source calibration composition:')
    print(focal_sub.to_string())
    print(f'  n subtypes {len(focal_sub)} | total {tot} | '
          f'largest "{focal_sub.index[0]}" {top} = {top/tot:.1%}')
    tgt_sub = (nsl_test[nsl_test['label'] == FOCAL_CLASS]
               .groupby('subtype').size().sort_values(ascending=False))
    shared = set(focal_sub.index) & set(tgt_sub.index)
    shared_mass = int(tgt_sub[list(shared)].sum()) if shared else 0
    print(f'  target {FOCAL_CLASS} mass in subtypes present in S_pool: '
          f'{shared_mass}/{int(tgt_sub.sum())} = {shared_mass/tgt_sub.sum():.1%}')

missing = (set(nsl_train['subtype']) - set(S_pool['subtype']))
print('\nsubtypes present in source pool but absent from S_pool:',
      sorted(missing) if missing else 'none')

 label         subtype  n_in_S_pool
   DoS         neptune         6180
   DoS           smurf          381
   DoS            back          148
   DoS        teardrop          148
   DoS             pod           25
   DoS            land            7
Normal          normal        10101
 Probe           satan          583
 Probe         ipsweep          540
 Probe       portsweep          413
 Probe            nmap          212
   R2L     warezclient          129
   R2L    guess_passwd           11
   R2L       ftp_write            3
   R2L     warezmaster            3
   R2L        multihop            2
   R2L            imap            1
   U2R buffer_overflow            5
   U2R            perl            1
   U2R         rootkit            1

FOCAL CLASS (R2L) source calibration composition:
subtype
warezclient     129
guess_passwd     11
ftp_write         3
warezmaster       3
multihop          2
imap              1
  n subtypes 6 | total 149 | largest "warezclient" 129 = 86.6%
  

In [14]:
import subprocess, re
from pathlib import Path

def sh(*args):
    r = subprocess.run(list(args), capture_output=True, text=True)
    print('$', ' '.join(args))
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print('stderr:', r.stderr.strip())
    print('exit:', r.returncode, '\n')
    return r

sh('git', 'remote', '-v')
sh('git', 'config', '--get', 'credential.helper')
sh('git', 'status', '-sb')

cred = Path('/root/.git-credentials')
print('.git-credentials present:', cred.exists())
if cred.exists():
    t = cred.read_text().strip()
    print('format ok:', bool(re.match(r'https://[^:]+:[^@]+@github\.com', t)))
    print('masked  :', re.sub(r':[^@]+@', ':****@', t))
print()
sh('git', 'push', '-v')

$ git remote -v
origin	https://github.com/anasbiswas1/calshift-research.git (fetch)
origin	https://github.com/anasbiswas1/calshift-research.git (push)
exit: 0 

$ git config --get credential.helper
exit: 1 

$ git status -sb
## main...origin/main [gone]
 M notebooks/02_partitions_and_feasibility.ipynb
exit: 0 

.git-credentials present: False

$ git push -v
stderr: Pushing to https://github.com/anasbiswas1/calshift-research.git
fatal: could not read Username for 'https://github.com': No such device or address
exit: 128 



CompletedProcess(args=['git', 'push', '-v'], returncode=128, stdout='', stderr="Pushing to https://github.com/anasbiswas1/calshift-research.git\nfatal: could not read Username for 'https://github.com': No such device or address\n")

In [15]:
import os, glob, shutil, subprocess
from pathlib import Path

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=True)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=True)
subprocess.run(['git','config','--global','credential.helper','store'], check=True)

hits = [p for p in glob.glob('/content/drive/MyDrive/**/.git-credentials', recursive=True)]
hits += [p for p in glob.glob('/content/drive/MyDrive/**/git-credentials', recursive=True)]
print('found credential files:', hits or 'NONE')

if hits:
    src = max(hits, key=lambda p: os.path.getmtime(p))
    print('using:', src)
    shutil.copy(src, '/root/.git-credentials')
    os.chmod('/root/.git-credentials', 0o600)

    dest = Path('/content/drive/MyDrive/.gitcreds'); dest.mkdir(parents=True, exist_ok=True)
    shutil.copy('/root/.git-credentials', dest / '.git-credentials')
    shutil.copy(Path.home() / '.gitconfig', dest / '.gitconfig')
    print('mirrored to', dest)

    r = subprocess.run(['git','push','-u','origin','main'], capture_output=True, text=True)
    print(r.stdout or '', r.stderr or '', 'exit:', r.returncode)
else:
    print('no credentials on Drive. Run the getpass cell below instead.')

found credential files: ['/content/drive/MyDrive/XIDS_Research/.git-credentials', '/content/drive/MyDrive/PICALIB_Research/.git-credentials', '/content/drive/MyDrive/CDTS_Research/.git-credentials', '/content/drive/MyDrive/CMED_Research/.git-credentials', '/content/drive/MyDrive/IOMT_Compression_Research/.git-credentials', '/content/drive/MyDrive/UAV_TRUST_Research/.git-credentials']
using: /content/drive/MyDrive/UAV_TRUST_Research/.git-credentials
mirrored to /content/drive/MyDrive/.gitcreds
Branch 'main' set up to track remote branch 'main' from 'origin'.
 To https://github.com/anasbiswas1/calshift-research.git
 * [new branch]      main -> main
 exit: 0


In [16]:
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, d in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
             ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb02: partitions and feasibility')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)


[main 921eae4] notebooks: credential discovery, non-raising git
 2 files changed, 2 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/01_setup_and_acquisition.ipynb
To https://github.com/anasbiswas1/calshift-research.git
   8b12467..921eae4  main -> main


CompletedProcess(args=['git', 'push'], returncode=0, stdout='', stderr='To https://github.com/anasbiswas1/calshift-research.git\n   8b12467..921eae4  main -> main\n')